## Лабораторная работа 7: Дискретное преобразование Фурье

### Упражнение 7.1: Реализация алгоритма FFT

In [ ]:
import sys
sys.path.insert(0, '../ThinkDSP/code')

import numpy as np
import matplotlib.pyplot as plt
from thinkdsp import decorate

PI2 = 2 * np.pi

### Лемма Даниэльсона-Ланцоша

Ключ к FFT - разделение массива на четные и нечетные элементы:

$$DFT(y)[n] = DFT(e)[n] + \exp(-2 \pi i n / N) DFT(o)[n]$$

где $e$ - четные элементы, $o$ - нечетные элементы.

### Тестовый сигнал

In [ ]:
# Простой тестовый массив
ys = [-0.5, 0.1, 0.7, -0.1]
hs = np.fft.fft(ys)
print('Эталонный результат (np.fft.fft):')
print(hs)

### Реализация DFT через матричное умножение

Для сравнения - простая реализация со сложностью O(N²).

In [ ]:
def dft(ys):
    """
    Вычисление DFT через матричное умножение.
    Сложность: O(N²)
    """
    N = len(ys)
    ts = np.arange(N) / N
    freqs = np.arange(N)
    args = np.outer(ts, freqs)
    M = np.exp(1j * PI2 * args)
    amps = M.conj().transpose().dot(ys)
    return amps

hs2 = dft(ys)
print('\nРезультат DFT:')
print(hs2)
print(f'Ошибка: {np.sum(np.abs(hs - hs2)):.2e}')

### Нерекурсивная версия FFT

Сначала реализуем версию, которая использует np.fft.fft для половин массива.

In [ ]:
def fft_norec(ys):
    """
    Нерекурсивная версия FFT для демонстрации принципа.
    """
    N = len(ys)
    
    # Разделяем на четные и нечетные элементы
    He = np.fft.fft(ys[::2])
    Ho = np.fft.fft(ys[1::2])
    
    # Весовые коэффициенты
    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)
    
    # Применяем лемму Даниэльсона-Ланцоша
    return np.tile(He, 2) + W * np.tile(Ho, 2)

hs3 = fft_norec(ys)
print('Результат fft_norec:')
print(hs3)
print(f'Ошибка: {np.sum(np.abs(hs - hs3)):.2e}')

### Рекурсивная реализация FFT

Полная реализация алгоритма FFT со сложностью O(N log N).

In [ ]:
def fft(ys):
    """
    Рекурсивная реализация FFT.
    Сложность: O(N log N)
    """
    N = len(ys)
    
    # Базовый случай
    if N == 1:
        return ys
    
    # Рекурсивные вызовы для четных и нечетных элементов
    He = fft(ys[::2])
    Ho = fft(ys[1::2])
    
    # Весовые коэффициенты
    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)
    
    # Комбинируем результаты
    return np.tile(He, 2) + W * np.tile(Ho, 2)

hs4 = fft(ys)
print('Результат рекурсивного FFT:')
print(hs4)
print(f'Ошибка: {np.sum(np.abs(hs - hs4)):.2e}')

**Комментарий:** Алгоритм FFT использует принцип "разделяй и властвуй", разбивая задачу на подзадачи меньшего размера. Это снижает сложность с O(N²) до O(N log N).

### Сравнение производительности

In [ ]:
import time

# Тестируем на массивах разного размера
sizes = [2**i for i in range(4, 11)]  # от 16 до 1024
times_dft = []
times_fft = []
times_numpy = []

for N in sizes:
    test_signal = np.random.randn(N)
    
    # DFT
    start = time.time()
    dft(test_signal)
    times_dft.append(time.time() - start)
    
    # Наш FFT
    start = time.time()
    fft(test_signal)
    times_fft.append(time.time() - start)
    
    # NumPy FFT
    start = time.time()
    np.fft.fft(test_signal)
    times_numpy.append(time.time() - start)

# Визуализация
plt.figure(figsize=(10, 6))
plt.loglog(sizes, times_dft, 'o-', label='DFT (O(N²))')
plt.loglog(sizes, times_fft, 's-', label='FFT рекурсивный (O(N log N))')
plt.loglog(sizes, times_numpy, '^-', label='NumPy FFT')
plt.xlabel('Размер массива N')
plt.ylabel('Время (с)')
plt.title('Сравнение производительности алгоритмов')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Комментарий:** FFT значительно быстрее DFT для больших массивов. NumPy FFT еще быстрее благодаря оптимизированной реализации на C.

### Проверка на более сложном сигнале

In [ ]:
# Создаем сигнал из нескольких частот
N = 64
t = np.linspace(0, 1, N)
signal = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 10 * t)

# Сравниваем результаты
result_numpy = np.fft.fft(signal)
result_our = fft(signal)

# Визуализация
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Исходный сигнал
axes[0].plot(t, signal)
axes[0].set_title('Исходный сигнал')
axes[0].set_xlabel('Время (с)')
axes[0].set_ylabel('Амплитуда')
axes[0].grid(True, alpha=0.3)

# Спектр
freqs = np.fft.fftfreq(N, 1/N)
axes[1].plot(freqs[:N//2], np.abs(result_our[:N//2]), 'o-', label='Наш FFT')
axes[1].plot(freqs[:N//2], np.abs(result_numpy[:N//2]), 'x--', label='NumPy FFT', alpha=0.7)
axes[1].set_title('Спектр')
axes[1].set_xlabel('Частота (Гц)')
axes[1].set_ylabel('Амплитуда')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Максимальная ошибка: {np.max(np.abs(result_numpy - result_our)):.2e}')

**Комментарий:** Наша реализация FFT дает результаты, идентичные NumPy FFT, но работает медленнее из-за накладных расходов Python. В реальных приложениях используют оптимизированные библиотеки.